# Light-sheet → deskew → OME-Zarr (single timepoint)
Playground based on `acquire_and_deskew.ipynb`. Acquires one LightSheetManager volume, deskews every channel on the GPU **in memory**, then writes a single OME-Zarr with the `OMEZarrImage` / `OMEZarrMultiscale` API.

Raw LSM data (ND-TIFF) still lands on the local `SAVE_DIRECTORY`; only the deskewed OME-Zarr goes to the network folder.

## 1. Connect

In [ ]:
import json, math, time
from pathlib import Path

import numpy
import matplotlib.pyplot as plt
import tifffile as tff
import pyclesperanto_prototype as cle
from pyclesperanto_prototype._tier8._affine_transform import _determine_translation_and_bounding_box
from ndtiff import Dataset

from pycromanager import Core
from lsm_pycromanager import LightSheetManager

core = Core()
lsm_mgr = LightSheetManager()
lsm = lsm_mgr.open()

print('GPU        :', cle.get_device())
print('Core camera:', core.get_camera_device())
print('LSM camera :', lsm.devices().first_active_camera_name())

## 2. Parameters (single timepoint)

In [ ]:
# ================= ACQUISITION =================
SLICES_PER_VIEW    = 101     # planes per volume
SLICE_STEP_UM      = 1.0     # um between planes (the raw galvo step)
MINIMIZE_SLICE_PERIOD = True # let LSM fit the period to the exposure

# ---- Camera ROI ----
# int -> centred square; (w, h) -> centred rectangle. None = full frame.
# Sensor is 1200x1200 binned (2400 unbinned at 2x2).
#
# IMPORTANT: ROI *height* caps the exposure. In PSEUDO_OVERLAP the camera reads
# out row by row, so the slice budget is roughly:
#       budget_ms = roi_height * LINE_SCAN_US / 1000
# and MicroManager rejects the acquisition if
#       cameraExposure + readout > budget
# where cameraExposure = SAMPLE_EXPOSURE_MS + ~8.5 ms of scan/camera delay.
# Cropping 1200 -> 600 rows halves the budget from ~49 ms to ~24.5 ms, which is
# why a 20 ms exposure that worked at full frame fails at 600 px.
ROI_SIZE_PX = 600            # centred crop -> ~4x less data (acquisition,
                             # deskew, save). Binning stays 2x2 (untouched).
                             # 600 rows -> 24.5 ms slice budget; exposure must
                             # be <= 15.75 ms, so 10 ms below is fine.
ROI_CAMERAS = ['Kinetix22-1', 'Kinetix22-2']

LINE_SCAN_US       = 40.83   # per binned row, at 100MHz 12bit / 2x2 binning
CAMERA_DELAY_MS    = 0.0     # with Minimize slice period, cameraExposure ~= sampleExposure
READOUT_MS         = 0.25

SAMPLE_EXPOSURE_MS = 20.0    # fits 600px with Minimize slice period (cam exp ~= sample)

# ---- Time series ----
USE_TIME_POINTS = False     # single-timepoint playground
NUM_TIME_POINTS = 1
TIME_INTERVAL_S = 7.0        # seconds between volumes (0 is rejected by LSM)

USE_CHANNELS  = False
CHANNEL_GROUP = 'Channel+Filter'
CHANNELS      = ['488+561']

SAVE_DIRECTORY   = r'C:\Users\aifadmin\Desktop\TEST'
SAVE_NAME_PREFIX = 'ls_single'
# Save format is left to the LightSheetManager GUI. The live deskew reads ND-TIFF.

# ================= LASERS =================
LASER_VOLTAGES = {
    '405': None, '445': None, '488': 0.5,
    '515': None, '561': 0.5,  '638': 0.5,
}

# ================= DESKEW =================
# Matches ASI_NEW_soLISH_20241219_lowmem.ipynb
lightsheet_angle_in_degrees = 40.5
time_vs_track  = 1
scaling_factor = 1
max_width      = 2048
dual_camera    = 1
galvo_vs_stage = 1

# LSM interleaves cameras across the channel axis ([cam1, cam2, cam1, cam2]),
# unlike the OME-TIFF notebooks which assume camera-major order, so the flip is
# decided by the recorded camera NAME rather than by channel index.
FLIP_CAMERA = 'Kinetix22-1'

selected_channels = 0
channelsList      = [0, 2, 3]
selected_positions = 0
posList            = [1, 2]
timepoints_start = 0
timepoints_range = 0

Z_max_projections = 1
Y_max_projections = 0
data3D_save       = 1
data_type         = numpy.uint16

VOXEL_XY_OVERRIDE = None

# Deskew shear direction depends on the galvo scan direction. The soLISH script
# uses lightsheet_angle directly for galvo data; this LSM data sweeps the other
# way, so it needs the supplementary angle (180 - angle). Flip this if the volume
# comes out skewed the wrong way. sin(180-a) == sin(a), so Z scaling is unchanged.
DESKEW_INVERT_ANGLE = False

deskewing_angle_in_degrees = (lightsheet_angle_in_degrees if galvo_vs_stage
                              else 180 - lightsheet_angle_in_degrees)
if DESKEW_INVERT_ANGLE:
    deskewing_angle_in_degrees = 180 - deskewing_angle_in_degrees
corr_value = numpy.sin(deskewing_angle_in_degrees / 180 * math.pi)
voxel_z_um = SLICE_STEP_UM / corr_value

# Volume depth comes from the settings, never from the axes observed on disk —
# during acquisition the z axis grows as slices land, so a half-written volume
# would otherwise look complete.
EXPECTED_NZ = SLICES_PER_VIEW

# ---- exposure budget preflight ----
_roi = ROI_SIZE_PX
_h = None if _roi is None else (_roi[1] if isinstance(_roi, (tuple, list)) else _roi)
if _h:
    budget_ms  = _h * LINE_SCAN_US / 1000.0
    max_sample = budget_ms - READOUT_MS - CAMERA_DELAY_MS
else:
    budget_ms, max_sample = None, None

print(f'Volume       : {SLICES_PER_VIEW} x {SLICE_STEP_UM} um '
      f'= {SLICES_PER_VIEW * SLICE_STEP_UM:.1f} um deep')
print(f'ROI          : {_roi if _roi else "full frame"}'
      f'{f"  -> slice budget {budget_ms:.1f} ms" if budget_ms else ""}')
print(f'Exposure     : {SAMPLE_EXPOSURE_MS} ms'
      f'{f"  (max {max_sample:.2f} ms for this ROI)" if max_sample else ""}')
print(f'Time points  : {NUM_TIME_POINTS if USE_TIME_POINTS else 1}'
      f'{f" every {TIME_INTERVAL_S} s" if USE_TIME_POINTS else ""}')
print(f'Deskew angle : {deskewing_angle_in_degrees} deg   -> voxel Z = {voxel_z_um:.4f} um')
print(f'Flip camera  : {FLIP_CAMERA if dual_camera else "(dual_camera off)"}')

if max_sample is not None and SAMPLE_EXPOSURE_MS > max_sample:
    need_h = int(numpy.ceil((SAMPLE_EXPOSURE_MS + CAMERA_DELAY_MS + READOUT_MS)
                            / LINE_SCAN_US * 1000))
    print(f'\n*** SAMPLE_EXPOSURE_MS={SAMPLE_EXPOSURE_MS} EXCEEDS the '
          f'{max_sample:.2f} ms budget for a {_h}-row ROI.')
    print(f'    MicroManager will refuse the acquisition. Either:')
    print(f'      - lower SAMPLE_EXPOSURE_MS to <= {max_sample:.2f}, or')
    print(f'      - use a taller ROI: ROI_SIZE_PX = ({_roi[0] if isinstance(_roi,(tuple,list)) else _roi}, {need_h})')

# ================= OME-ZARR OUTPUT =================
# Raw LSM data (ND-TIFF) goes to SAVE_DIRECTORY (local, fast). The deskewed
# volume is written as OME-Zarr here.
OMEZARR_DIR  = Path(r'Y:\Collaborations\aif-botton-collaboration\Johannes\20260817\ome zarr test\lightsheet')
OMEZARR_DIR.mkdir(parents=True, exist_ok=True)
OMEZARR_NAME = 'ls_deskewed'

# Deskewed output voxel size (isotropic). cle.deskew_y resamples to isotropic
# voxels at the lateral pixel size; None -> use the lateral pixel size the
# deskew actually used (read from the data). Verify against a known structure.
DESKEW_OUT_VOXEL_UM = None

print(f'OME-Zarr out : {OMEZARR_DIR}')


## 3. Apply acquisition settings

In [ ]:
# LightSheetManager settings keys changed in newer plugin versions: the trailing
# '_' was dropped, timing fields gained an 'Ms' suffix, and timePointInterval_ was
# renamed timePointIntervalSec. These helpers read/write whichever names exist, so
# the notebook works on both old and new LSM.
def _get(d, *names, default=None):
    for n in names:
        if n in d:
            return d[n]
    return default


def _set(d, value, *names):
    for n in names:
        if n in d:
            d[n] = value
            return n
    d[names[0]] = value
    return names[0]


def apply_lsm_settings(lsm):
    """Write our parameters into the LSM settings in one atomic update.

    Mutates the current settings JSON in place (using whatever key names it
    already has) rather than constructing a fixed patch, so a plugin field rename
    can't silently drop a setting.
    """
    settings = lsm.acquisitions().settings()
    cls = settings.get_class()
    if 'java.lang.Class' not in cls._interfaces:
        cls._interfaces.append('java.lang.Class')
    j = json.loads(settings.to_pretty_json())

    vol = _get(j, 'volume', 'volume_', default={})
    sl  = _get(j, 'slice', 'slice_', default={})
    _set(vol, int(SLICES_PER_VIEW), 'slicesPerView', 'slicesPerView_')
    _set(vol, float(SLICE_STEP_UM), 'sliceStepSize', 'sliceStepSize_')
    _set(sl,  float(SAMPLE_EXPOSURE_MS), 'sampleExposure', 'sampleExposure_')
    _set(sl,  bool(MINIMIZE_SLICE_PERIOD), 'periodMinimized', 'periodMinimized_')
    _set(j, False, 'useAdvancedTiming', 'useAdvancedTiming_')  # honor Minimize slice period
    _set(j, bool(USE_TIME_POINTS), 'useTimePoints', 'useTimePoints_')
    _set(j, int(NUM_TIME_POINTS), 'numTimePoints', 'numTimePoints_')
    _set(j, float(TIME_INTERVAL_S), 'timePointIntervalSec', 'timePointInterval_')
    _set(j, True, 'saveDuringAcq', 'saveDuringAcq_')
    _set(j, SAVE_DIRECTORY, 'saveDirectory', 'saveDirectory_')
    _set(j, SAVE_NAME_PREFIX, 'saveNamePrefix', 'saveNamePrefix_')
    if USE_CHANNELS:
        ch = _get(j, 'channels', 'channels_', default={})
        _set(ch, True, 'enabled', 'enabled_')
        _set(ch, 'VOLUME', 'mode', 'mode_')
        _set(ch, CHANNEL_GROUP, 'group', 'group_')
        _set(ch, {CHANNEL_GROUP: [
                  {'useChannel_': True, 'group_': CHANNEL_GROUP, 'name_': c, 'offset_': 0.0}
                  for c in CHANNELS]}, 'groups', 'groups_')

    lsm.acquisitions().update_settings(settings.from_json(json.dumps(j), cls))


# ---------- camera ROI ----------
def full_frame(cam):
    """Full binned frame size, from the chip dimensions and current binning."""
    xdim = int(float(core.get_property(cam, 'X-dimension')))
    ydim = int(float(core.get_property(cam, 'Y-dimension')))
    b = int(str(core.get_property(cam, 'Binning')).split('x')[0])
    return xdim // b, ydim // b


def apply_roi(cam, size):
    """Centre an ROI; int -> square, (w,h) -> rect, None -> full frame.

    Computes the centre from the chip dimensions rather than clearing first:
    clear_roi() takes no camera label (so it would force switching the Core
    camera) and restoring the full frame costs ~7 s per camera.
    """
    fw, fh = full_frame(cam)
    if size is None:
        target = (0, 0, fw, fh)
    else:
        w, h = size if isinstance(size, (tuple, list)) else (size, size)
        target = ((fw - w) // 2, (fh - h) // 2, w, h)
    r = core.get_roi(cam)
    if (r.x, r.y, r.width, r.height) == target:
        return False                       # already correct, skip the reconfigure
    core.set_roi(cam, *(int(v) for v in target))
    return True


for cam in ROI_CAMERAS:
    try:
        t_roi = time.time()
        changed = apply_roi(cam, ROI_SIZE_PX)
        r = core.get_roi(cam)
        print(f'ROI {cam:14s}: x={r.x} y={r.y} w={r.width} h={r.height}'
              f'  ({"set in %.2fs" % (time.time()-t_roi) if changed else "already set"})')
    except Exception as e:
        print(f'ROI {cam:14s}: FAILED ({e})')

# ---------- lasers / shutter ----------
for line, volts in LASER_VOLTAGES.items():
    if volts is None:
        continue
    try:
        core.set_property(line, 'Voltage', str(float(volts)))
    except Exception as e:
        print(f'  laser {line}: FAILED ({e})')

# LightSheetManager gates the lasers through the shutter during its sweep, so
# auto-shutter MUST be on. With it off the lasers never fire and the volume is
# just read noise.
core.set_auto_shutter(True)

print('Laser voltages :', {k: core.get_property(k, 'Voltage')
                           for k in LASER_VOLTAGES if LASER_VOLTAGES[k] is not None})
print('Auto-shutter   :', core.get_auto_shutter(), '(must be True for LSM)')
print('PLogic channel :', core.get_property('PLogic:E:36', 'OutputChannel'))

# ---------- acquisition settings (single atomic update, schema-agnostic) ----------
apply_lsm_settings(lsm)

s   = json.loads(lsm.acquisitions().settings().to_pretty_json())
vol = _get(s, 'volume', 'volume_', default={})
tim = _get(s, 'timing', 'timing_', default={})
sl  = _get(s, 'slice', 'slice_', default={})
n        = _get(vol, 'slicesPerView', 'slicesPerView_')
d        = _get(tim, 'sliceDurationMs', 'sliceDuration_')
camexp   = _get(tim, 'cameraExposureMs', 'cameraExposure_')
stepsize = _get(vol, 'sliceStepSize', 'sliceStepSize_')
sampexp  = _get(sl, 'sampleExposure', 'sampleExposure_')
periodmin = _get(sl, 'periodMinimized', 'periodMinimized_')
uses_tp  = _get(s, 'useTimePoints', 'useTimePoints_')
nt       = _get(s, 'numTimePoints', 'numTimePoints_') if uses_tp else 1
interval = _get(s, 'timePointIntervalSec', 'timePointInterval_') or 0.0
savemode = _get(s, 'saveMode', 'saveMode_')

print()
print(f"slicesPerView   : {n}")
print(f"sliceStepSize   : {stepsize} um")
print(f"sampleExposure  : {sampexp} ms")
print(f"periodMinimized : {periodmin}")
print(f"sliceDuration   : {d} ms")
print(f"cameraExposure  : {camexp} ms")
print(f"timePoints      : {nt}  (interval {interval} s)")
print(f"saveMode        : {savemode}")
print(f"\n-> {n * d / 1000:.2f} s per volume x {nt} = "
      f"{n * d / 1000 * nt + max(0, nt - 1) * interval:.1f} s total")

problems = []
if int(n) != EXPECTED_NZ:
    problems.append(f"slicesPerView {n} != EXPECTED_NZ {EXPECTED_NZ} "
                    f"- set SLICES_PER_VIEW={int(n)} and re-run the parameters cell")
if not periodmin and MINIMIZE_SLICE_PERIOD:
    problems.append("periodMinimized did not stick - the exposure error will return")
if int(nt) != (NUM_TIME_POINTS if USE_TIME_POINTS else 1):
    problems.append(f"timePoints {nt} != requested {NUM_TIME_POINTS}")
if savemode != 'ND_TIFF':
    problems.append("saveMode is not ND_TIFF - the deskew reader expects ND-TIFF")

_r = core.get_roi(ROI_CAMERAS[0])
_budget = _r.height * LINE_SCAN_US / 1000.0
if camexp is not None and camexp + READOUT_MS > _budget:
    problems.append(
        f"cameraExposure {camexp} + {READOUT_MS} exceeds the {_budget:.1f} ms "
        f"line-scan budget for a {_r.height}-row ROI - lower SAMPLE_EXPOSURE_MS "
        f"or use a taller ROI")
else:
    print(f"exposure budget : {camexp + READOUT_MS:.2f} / {_budget:.1f} ms  ({_r.height} rows)  OK")
for p in problems:
    print(f"\nWARNING: {p}")

## 4. Deskew engine

In [ ]:
class Deskewer:
    """Holds the GPU buffers and deskews one (position, time, channel) volume."""

    def __init__(self, n_z, height, width, vxy, vz):
        self.n_z, self.h, self.w = n_z, height, width
        self.vx = self.vy = float(vxy)
        self.vz = float(vz)

        transform = cle.AffineTransform3D()
        transform._deskew_y(angle_in_degrees=deskewing_angle_in_degrees,
                            voxel_size_x=self.vx, voxel_size_y=self.vy,
                            voxel_size_z=self.vz, scale_factor=scaling_factor)
        probe = cle.create((n_z, height, width))
        self.new_size, _, _ = _determine_translation_and_bounding_box(probe, transform)
        self.mw = min(max_width, self.new_size[2])

        self.deskewed_ = cle.create([self.new_size[0], self.new_size[1], self.mw])
        self.max_z_    = cle.create([self.new_size[1], self.mw])
        self.max_y_    = cle.create([self.mw, self.new_size[0]])
        self.data_rot  = cle.create([n_z, height, self.mw])

        self.deskewed = numpy.zeros(self.new_size, dtype=data_type)
        self.max_z    = numpy.zeros(self.new_size[1:3], dtype=data_type)
        self.max_y    = numpy.zeros([self.new_size[2], self.new_size[0]], dtype=data_type)

    def run(self, image3D, flip):
        if flip:
            for zz in range(len(image3D)):
                image3D[zz] = numpy.fliplr(image3D[zz])

        num_steps = int(numpy.ceil(image3D.shape[2] / self.mw))
        delta = self.mw
        for ll in range(num_steps):
            lo, hi = ll * self.mw, ll * self.mw + self.mw
            hi_inv = image3D.shape[2] - ll * self.mw
            lo_inv = hi_inv - self.mw
            if hi > image3D.shape[2]:
                hi, lo_inv, delta = image3D.shape[2], 0, image3D.shape[2] - lo

            if galvo_vs_stage == 0:
                cle.rotate(image3D[:, :, lo:hi], self.data_rot,
                           angle_around_z_in_degrees=180, rotate_around_center=True)
                src = self.data_rot
            else:
                src = image3D[:, :, lo:hi]

            cle.deskew_y(src, self.deskewed_,
                         angle_in_degrees=deskewing_angle_in_degrees,
                         voxel_size_x=self.vx, voxel_size_y=self.vy, voxel_size_z=self.vz,
                         linear_interpolation=True, scale_factor=scaling_factor)
            self.deskewed[:, :, lo_inv:hi_inv] = self.deskewed_[:, :, 0:delta]

            if Z_max_projections:
                cle.maximum_z_projection(self.deskewed_, self.max_z_)
                self.max_z[:, lo_inv:hi_inv] = self.max_z_[:, 0:delta]
            if Y_max_projections:
                cle.maximum_y_projection(self.deskewed_, self.max_y_)
                self.max_y[lo_inv:hi_inv, :] = self.max_y_[0:delta, :]

        return self.deskewed, self.max_z, self.max_y


def axes_of(ds):
    return {k: sorted(v) for k, v in ds.axes.items()}


def coords_for(ax, p, t, c, z=None):
    d = {}
    if 'position' in ax: d['position'] = p
    if 'time'     in ax: d['time'] = t
    if 'channel'  in ax: d['channel'] = c
    if z is not None:    d['z'] = z
    return d


def camera_map(ds, ax, p, t):
    """channel index -> camera name, from the per-image metadata.

    LSM interleaves cameras across channels ([cam1, cam2, cam1, cam2]) whereas the
    OME-TIFF notebooks assume camera-major order ([cam1, cam1, cam2, cam2]).
    Reading the recorded Camera name is order-independent and can't get this wrong.
    """
    out = {}
    for c in ax.get('channel', [0]):
        try:
            out[c] = ds.read_metadata(**coords_for(ax, p, t, c, 0)).get('Camera')
        except Exception:
            out[c] = None
    return out


def volume_ready(ds, ax, p, t, c, n_z=None):
    """True once ALL expected z planes of this (p,t,c) are on disk.

    n_z comes from the acquisition settings, never from ds.axes — during a live
    acquisition the observed z axis grows as slices arrive, so checking against
    it would report a half-written volume as complete.
    """
    n_z = EXPECTED_NZ if n_z is None else n_z
    # Fast gate: slices are written in order, so if the last one is missing the
    # volume can't be complete. Saves ~100 index lookups per poll per channel.
    if not ds.has_image(**coords_for(ax, p, t, c, n_z - 1)):
        return False
    for z in range(n_z):
        if not ds.has_image(**coords_for(ax, p, t, c, z)):
            return False
    return True


def read_volume(ds, ax, p, t, c, n_z, h, w, dtype):
    out = numpy.empty((n_z, h, w), dtype=dtype)
    for z in range(n_z):
        out[z] = ds.read_image(**coords_for(ax, p, t, c, z))
    return out


print(f'deskew engine ready  (expecting {EXPECTED_NZ} slices per volume)')
print(f'flip camera        : {FLIP_CAMERA}')

## 5. Acquire + deskew (in memory)
Runs one volume, waits for the writer to finish, then deskews each channel into RAM. **Run step 2's widefield/LSM setup as needed and park/focus the sample first.**

In [ ]:
import contextlib, io

POLL_S    = 0.5
TIMEOUT_S = 1800
QUIET_S   = 5.0      # index stable this long => writer finished


@contextlib.contextmanager
def quiet():
    with contextlib.redirect_stdout(io.StringIO()):
        yield


save_root = Path(SAVE_DIRECTORY)
before = {p.name for p in save_root.glob(f'{SAVE_NAME_PREFIX}*') if p.is_dir()}

t0 = time.time()
lsm.acquisitions().request_run()
print('Acquisition requested (single timepoint)...')

# --- new dataset folder ---
run_dir = None
while time.time() - t0 < 120:
    new = {p.name for p in save_root.glob(f'{SAVE_NAME_PREFIX}*') if p.is_dir()} - before
    cand = [save_root / n for n in new if (save_root / n / 'NDTiff.index').exists()]
    if cand:
        run_dir = max(cand, key=lambda p: p.stat().st_mtime)
        break
    time.sleep(POLL_S)
if run_dir is None:
    raise RuntimeError('No new ND-TIFF dataset appeared in 120 s - check the LSM window / save mode.')
print('Dataset:', run_dir.name)

# --- wait for the writer to finish (only stat the index; nothing contends with LSM) ---
index_file = run_dir / 'NDTiff.index'
last_size, last_growth = -1, time.time()
while time.time() - t0 < TIMEOUT_S:
    size = index_file.stat().st_size if index_file.exists() else 0
    if size != last_size:
        last_size, last_growth = size, time.time()
    if size > 0 and (time.time() - last_growth) > QUIET_S:
        break
    time.sleep(POLL_S)
print(f'Writer finished after {time.time()-t0:.1f}s - deskewing in memory...')

# --- deskew every channel of the (single) position/timepoint, keep volumes in RAM ---
with quiet():
    ds = Dataset(str(run_dir))
dsk = None
channels, maxproj, cams = {}, {}, {}
try:
    ax = axes_of(ds)
    p = ax.get('position', [0])[0]
    t = ax.get('time', [0])[0]
    cams = camera_map(ds, ax, p, t)
    print('channel -> camera:', ', '.join(f'C{k}={v}' for k, v in cams.items()))

    for c in ax.get('channel', [0]):
        if selected_channels and c not in channelsList:
            continue
        if not volume_ready(ds, ax, p, t, c):
            print(f'  C{c}: incomplete, skipped'); continue
        c0 = coords_for(ax, p, t, c, 0)
        if dsk is None:
            probe = ds.read_image(**c0)
            mdimg = ds.read_metadata(**c0)
            vxy = VOXEL_XY_OVERRIDE or mdimg.get('PixelSizeUm')
            if not vxy:
                raise ValueError('No PixelSizeUm - set VOXEL_XY_OVERRIDE')
            dsk = Deskewer(EXPECTED_NZ, probe.shape[0], probe.shape[1], vxy, voxel_z_um)
            print(f'  voxel xy={dsk.vx} um  z={dsk.vz:.4f} um  -> output {dsk.new_size}')
        flip = bool(dual_camera and cams.get(c) == FLIP_CAMERA)
        img = read_volume(ds, ax, p, t, c, dsk.n_z, dsk.h, dsk.w, ds.dtype)
        tv = time.time()
        vol, mz, my = dsk.run(img, flip)
        channels[c] = vol.copy()     # copy: the Deskewer reuses one output buffer
        maxproj[c]  = mz.copy()
        print(f'  C{c} ({cams.get(c)}, flip={flip})  {vol.shape}  '
              f'max={int(vol.max())}  {time.time()-tv:.2f}s')
finally:
    with quiet():
        ds.close()

print(f'\nDeskewed {len(channels)} channel(s), each {dsk.new_size}, held in memory.')

# quick look: max-Z projection of each channel
if channels:
    ids = sorted(channels)
    fig, axes = plt.subplots(1, len(ids), figsize=(5 * len(ids), 5), squeeze=False)
    for j, c in enumerate(ids):
        m = maxproj[c].astype(numpy.float32)
        p1, p99 = numpy.percentile(m, (1, 99))
        axes[0][j].imshow(numpy.clip((m - p1) / (p99 - p1 + 1e-9), 0, 1), cmap='gray')
        axes[0][j].set_title(f'C{c} ({cams.get(c)})  max-Z'); axes[0][j].axis('off')
    plt.tight_layout(); plt.show()

## 6. Save as OME-Zarr
`OMEZarrImage` / `OMEZarrMultiscale` shim (same call style as ome-zarr 0.18, on the installed 0.16). Run the shim cell once, then the save cell.

In [ ]:
import numpy as np
# Compat shim: OMEZarrImage / OMEZarrMultiscale on top of ome-zarr 0.16 write_multiscale
import shutil
from pathlib import Path
import numpy as np
import zarr
from ome_zarr.io import parse_url
from ome_zarr.writer import write_multiscale

_UNIT = {'micrometer': 'micrometer', 'um': 'micrometer', 'µm': 'micrometer',
         'nanometer': 'nanometer', 'nm': 'nanometer',
         'millimeter': 'millimeter', 'mm': 'millimeter'}
_TYPE = {'x': 'space', 'y': 'space', 'z': 'space', 'c': 'channel', 't': 'time'}


class OMEZarrImage:
    """A labelled array: data + axis names + per-axis physical scale (+ units)."""
    def __init__(self, data, axes, scale, axes_units=None):
        self.data = np.asarray(data)
        self.axes = list(axes)
        self.scale = dict(scale)
        self.axes_units = dict(axes_units or {})
        if self.data.ndim != len(self.axes):
            raise ValueError(f'axes {self.axes} do not match data.ndim={self.data.ndim}')


class OMEZarrMultiscale:
    """Build a multiscale pyramid and write it as OME-Zarr.

    Mirrors the ome-zarr 0.18 API on top of the 0.16 write_multiscale. Pyramid
    downsampling is applied to the x/y axes only (z/c/t are kept), which is what
    a thin z-stack overview wants. `scale_factors` are absolute factors from full
    resolution, e.g. (2, 4, 8) -> levels [full, /2, /4, /8].
    """
    def __init__(self, image, scale_factors=(2, 4, 8), method='nearest',
                 channel_names=None, channel_colors=None, contrast_limits=None,
                 chunks=None):
        self.image = image
        self.scale_factors = tuple(scale_factors)
        self.method = method
        self.channel_names = channel_names
        self.channel_colors = channel_colors
        self.contrast_limits = contrast_limits
        self.chunks = chunks

    def _downsample(self, data, f):
        ax = self.image.axes
        if self.method == 'resize':
            from skimage.transform import resize
            shp = [(s // f if ax[i] in ('x', 'y') else s) for i, s in enumerate(data.shape)]
            return resize(data, shp, order=1, preserve_range=True,
                          anti_aliasing=False).astype(data.dtype)
        sl = tuple(slice(None, None, f) if ax[i] in ('x', 'y') else slice(None)
                   for i in range(data.ndim))
        return data[sl]

    def _pyramid(self):
        d = self.image.data
        return [d] + [self._downsample(d, f) for f in self.scale_factors]

    def _axes_meta(self):
        out = []
        for name in self.image.axes:
            a = {'name': name, 'type': _TYPE.get(name, 'space')}
            u = self.image.axes_units.get(name)
            if u:
                a['unit'] = _UNIT.get(u, u)
            out.append(a)
        return out

    def _transforms(self, pyramid):
        ax, full = self.image.axes, pyramid[0].shape
        return [[{'type': 'scale',
                  'scale': [self.image.scale[ax[i]] * (full[i] / lv.shape[i])
                            for i in range(len(ax))]}] for lv in pyramid]

    def _omero(self, name):
        ax = self.image.axes
        n = self.image.data.shape[ax.index('c')] if 'c' in ax else 1
        names = self.channel_names or [f'ch{i}' for i in range(n)]
        colors = self.channel_colors or ['FFFFFF'] * n
        chans = []
        for i in range(n):
            ch = {'label': names[i], 'color': colors[i % len(colors)], 'active': True}
            if self.contrast_limits:
                lo, hi = self.contrast_limits[i]
                ch['window'] = {'start': lo, 'end': hi, 'min': lo, 'max': hi}
            chans.append(ch)
        return {'name': name, 'channels': chans}

    def to_ome_zarr(self, path, overwrite=True):
        p = Path(path)
        if overwrite and p.exists():
            shutil.rmtree(p)
        pyr = self._pyramid()
        store = parse_url(str(p), mode='w').store
        root = zarr.group(store=store)
        so = dict(chunks=self.chunks) if self.chunks else None
        write_multiscale(pyramid=pyr, group=root, axes=self._axes_meta(),
                         coordinate_transformations=self._transforms(pyr),
                         storage_options=so)
        root.attrs['omero'] = self._omero(p.stem)
        return p

print('OMEZarrImage / OMEZarrMultiscale ready')

In [ ]:
# Assemble the deskewed channels -> (C, Z, Y, X) or (Z, Y, X) and save OME-Zarr.
chan_ids = sorted(channels)
if not chan_ids:
    raise RuntimeError('No deskewed channels in memory - run the acquire+deskew cell first.')

vx = float(dsk.vx if DESKEW_OUT_VOXEL_UM is None else DESKEW_OUT_VOXEL_UM)  # isotropic
_palette = ['00FFFF', 'FF00FF', 'FFFF00', 'FFFFFF', '00FF00', 'FF0000']

if len(chan_ids) == 1:
    data  = channels[chan_ids[0]]
    axes_ = ['z', 'y', 'x']
    scale = {'z': vx, 'y': vx, 'x': vx}
    chunks = (1, 256, 256)
else:
    data  = numpy.stack([channels[c] for c in chan_ids], axis=0)
    axes_ = ['c', 'z', 'y', 'x']
    scale = {'c': 1.0, 'z': vx, 'y': vx, 'x': vx}
    chunks = (1, 1, 256, 256)

units     = {'z': 'micrometer', 'y': 'micrometer', 'x': 'micrometer'}
ch_names  = [f'C{c} ({cams.get(c)})' for c in chan_ids]
ch_colors = [_palette[i % len(_palette)] for i in range(len(chan_ids))]

image = OMEZarrImage(data=data, axes=axes_, scale=scale, axes_units=units)
multiscales = OMEZarrMultiscale(
    image=image,
    scale_factors=(2, 4, 8),
    method='nearest',
    channel_names=ch_names,
    channel_colors=ch_colors,
    chunks=chunks,
)
OMEZARR_PATH = OMEZARR_DIR / f'{OMEZARR_NAME}.ome.zarr'
multiscales.to_ome_zarr(OMEZARR_PATH)

print(f'Saved OME-Zarr : {OMEZARR_PATH}')
print(f'  data {data.shape} {data.dtype}   isotropic voxel {vx:.4f} um')
print(f'  channels : {ch_names}')
print(f'  levels   : {[lv.shape for lv in multiscales._pyramid()]}')

## 7. Close

In [ ]:
lsm_mgr.close()
print('LSM closed.')
print('PLogic OutputChannel :', core.get_property('PLogic:E:36', 'OutputChannel'))
print('Auto-shutter         :', core.get_auto_shutter())